# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Praveen23-kk/FlyRank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I will construct the feature vector directly from `data/raw/content_refresh_anonymized.csv`. This includes filling NaNs for numeric and categorical columns, parsing the label `is_declining_label` from `trend_direction`, and transforming highly skewed metrics (like impressions) with `log1p`.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Fill missing values
numeric_cols = ["search_volume", "competition", "cpc", "word_count", "char_count", "impressions_90d", "clicks_90d", "sessions_90d"]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Categorical missing values
cat_cols = ["competition_level", "content_type", "main_intent"]
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].fillna("unknown").astype(str)

# Create label and drop label source
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Log features
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])

print(f"Built feature vector with {len(df)} rows. Example of target distribution:")
print(df["is_declining_label"].value_counts(normalize=True))

Built feature vector with 30000 rows. Example of target distribution:
is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

Here is a summary of the features I plan to use:
- **`search_volume`, `competition`, `cpc`**: Keyword context features. These are missing for non-keyword articles (like `feedly article`). Missing values are filled with 0. These exist before prediction.
- **`word_count`, `char_count`**: Content length. Missing values are filled with 0. These exist before prediction.
- **`impressions_90d`, `clicks_90d`, `sessions_90d`**: Trailing 90-day activity. Always present. Exist before prediction.
- **`content_age_days`, `days_since_last_update`**: Freshness. Exist before prediction.
- **`content_type`, `main_intent`**: Categoricals, one-hot encoded or handled by tree models. Available before prediction.

In [2]:
# Check missingness by content type to demonstrate why blind fillna(0) can be risky
missing_stats = df.groupby("content_type")[["search_volume", "word_count"]].apply(lambda x: x.isna().mean())
print("Missing rates by content_type:")
print(missing_stats)

Missing rates by content_type:
                    search_volume  word_count
content_type                                 
comparison article            0.0         0.0
feedly article                0.0         0.0
keyword article               0.0         0.0


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I will train two models to detect leakage:
1. **Model A (Leaky)**: Uses `trend_pct` (the column `trend_direction` is directly derived from) as a feature.
2. **Model B (Honest)**: Removes `trend_pct` and any future-overlapping windows like `impressions_last_30d` (which overlaps the label window).

A massive score drop from Model A to Model B confirms that `trend_pct` was a leaky feature giving away the answer.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Re-load clean to be safe
df_test = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df_test["is_declining_label"] = (df_test["trend_direction"] == "down").astype(int)

# 1. Leaky Model: Includes trend_pct
leaky_features = ["word_count", "impressions_90d", "trend_pct"]
X_leaky = df_test[leaky_features].copy()
X_leaky["trend_pct"] = X_leaky["trend_pct"].fillna(0)
X_leaky["word_count"] = X_leaky["word_count"].fillna(0)

y = df_test["is_declining_label"]

X_train, X_test, y_train, y_test = train_test_split(X_leaky, y, test_size=0.2, random_state=42)
clf = RandomForestClassifier(n_estimators=50, random_state=42)
clf.fit(X_train, y_train)
leaky_auc = roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])

# 2. Honest Model: Excludes trend_pct
honest_features = ["word_count", "impressions_90d"]
X_honest = df_test[honest_features].copy()
X_honest["word_count"] = X_honest["word_count"].fillna(0)

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_honest, y, test_size=0.2, random_state=42)
clf_h = RandomForestClassifier(n_estimators=50, random_state=42)
clf_h.fit(X_train_h, y_train_h)
honest_auc = roc_auc_score(y_test_h, clf_h.predict_proba(X_test_h)[:, 1])

print(f"Leaky Model AUC (with trend_pct): {leaky_auc:.4f}")
print(f"Honest Model AUC (without trend_pct): {honest_auc:.4f}")
print("The collapse in AUC confirms trend_pct is severely leaky and must be dropped.")

Leaky Model AUC (with trend_pct): 1.0000
Honest Model AUC (without trend_pct): 0.6042
The collapse in AUC confirms trend_pct is severely leaky and must be dropped.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- **`trend_direction`**: This is the literal source of the label.
- **`trend_pct`**: The label is derived directly from this value (`<-20` means down). This is label leakage.
- **`impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`**: The label is computed over the last 30 days vs previous 30 days. These columns overlap with the label window, so using them as features is future-window leakage.
- **`content_id`, `client_id`**: These are identifiers. Using them as features would just lead to memorization of specific pages or clients rather than learning generalizable patterns. They should only be used for grouping (e.g. `GroupKFold` on `client_id`).

In [4]:
excluded_cols = ["trend_direction", "trend_pct", "impressions_last_30d", "clicks_last_30d", "sessions_last_30d", "content_id", "client_id"]
print("Excluded columns to prevent leakage or memorization:")
for col in excluded_cols:
    print(f"- {col}")

Excluded columns to prevent leakage or memorization:
- trend_direction
- trend_pct
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- content_id
- client_id


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.